# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
from typing import List, Optional, Union
from openai import OpenAI
from IPython.display import display, Markdown

In [5]:
# configs
lamma_config = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "llama3.2",
    "model": "llama3.2"
}

gpt_mini_config = {
    "model": "gpt-4o-mini",
}

system_prompt = "You are experienced senior ML/AI Machine Learning Engineer with expertise in LLM engineering. \
    Your role is to assist and mentor junior ML/AI engineers in their tasks and answers technical questions. \
    Your final response should be in markdown format."

user_prompt = """
How would you add functinality to LLMClient class to support json output and markdown formatting?

class LLMClient:
    def __init__(self, config: dict):
        self.model = config.get("model", "llama3.2")
        
        self.client = OpenAI(
            base_url=config.get("base_url", 'http://localhost:11434/v1'),
            api_key=config.get("api_key", "ollama")
        )

    def chat(
            self,
            user_prompt: str,
            system_prompt: Optional[str]=None,
            messages: Optional[List[dict]]=None,
            stream: bool = False
    ) -> Union[str, List[str]]:
        
        if messages:
            chat_messages = messages
        else:
            chat_messages = []
            if system_prompt:
                chat_messages.append({"role": "system", "content": system_prompt})
            chat_messages.append({"role": "user", "content": user_prompt})
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=chat_messages,
                stream=stream
            )
            if stream:
                for chunk in response:
                    delta = chunk.choices[0].delta.content or ""
                    yield delta
            else:
                return response.choices[0].message.content.strip()
            
        except Exception as e:
            raise RuntimeError(f"LLM reqeust failed: {str(e)}")

"""

In [3]:
# utils
class LLMClient:
    def __init__(self, config: dict):
        self.model = config.get("model", "llama3.2")

        self.client = OpenAI(
            base_url=config.get("base_url", None),
            api_key=config.get("api_key", None),
        )

    def chat(
        self,
        user_prompt: str,
        system_prompt: Optional[str] = None,
        messages: Optional[List[dict]] = None,
        stream: bool = False,
    ) -> Union[str, List[str]]:

        if messages:
            chat_messages = messages
        else:
            chat_messages = []
            if system_prompt:
                chat_messages.append({"role": "system", "content": system_prompt})
            chat_messages.append({"role": "user", "content": user_prompt})

        try:
            response = self.client.chat.completions.create(
                model=self.model, messages=chat_messages, stream=stream
            )
            if stream:

                def stream_generator():
                    for chunk in response:
                        delta = chunk.choices[0].delta.content or ""
                        yield delta

                return stream_generator()
            else:
                return response.choices[0].message.content.strip()

        except Exception as e:
            raise RuntimeError(f"LLM reqeust failed: {str(e)}")


gpt_client = LLMClient(gpt_mini_config)
display(Markdown(gpt_client.chat(user_prompt=user_prompt, system_prompt=system_prompt)))

To add functionality for JSON output and Markdown formatting in the `LLMClient` class, we can:

1. Introduce parameters to specify the output format (JSON or Markdown).
2. Modify the `chat` method to format its output accordingly.

Here's how you can implement these functionalities:

```python
import json
from typing import List, Optional, Union, Dict, Any
from some_openai_library import OpenAI  # Replace with the actual import for the OpenAI client

class LLMClient:
    def __init__(self, config: dict):
        self.model = config.get("model", "llama3.2")
        self.client = OpenAI(
            base_url=config.get("base_url", 'http://localhost:11434/v1'),
            api_key=config.get("api_key", "ollama")
        )

    def chat(
            self,
            user_prompt: str,
            system_prompt: Optional[str] = None,
            messages: Optional[List[dict]] = None,
            stream: bool = False,
            format: str = 'text'  # New parameter for output format
    ) -> Union[str, List[str], Dict[str, Any]]:
        
        if messages:
            chat_messages = messages
        else:
            chat_messages = []
            if system_prompt:
                chat_messages.append({"role": "system", "content": system_prompt})
            chat_messages.append({"role": "user", "content": user_prompt})
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=chat_messages,
                stream=stream
            )
            if stream:
                for chunk in response:
                    delta = chunk.choices[0].delta.content or ""
                    # Yielding in specified format
                    yield self.format_output(delta, format) 
            else:
                content = response.choices[0].message.content.strip()
                return self.format_output(content, format)  # Format the final output
            
        except Exception as e:
            raise RuntimeError(f"LLM request failed: {str(e)}")

    def format_output(self, content: str, format: str) -> Union[str, Dict[str, Any]]:
        """Formats the output based on the specified format."""
        if format == 'json':
            return {
                "response": content,
                "model": self.model,
            }
        elif format == 'markdown':
            # Simple Markdown formatting for various uses
            return f"**LLM Response**:\n\n{content}\n"
        else:  # Default to plain text
            return content

# Example usage:
# client = LLMClient(config)
# response = client.chat("What is the weather today?", format='json')
# print(response)
```

### Changes Made:
1. **Added a `format` Parameter**: In the `chat` method, the `format` parameter specifies the output format (options: 'text', 'json', 'markdown').
2. **Format Output Method**: Created a `format_output` method to handle output formatting based on the specified format. 
   - For **JSON**, it returns a dictionary with the response and model used.
   - For **Markdown**, it wraps the response in Markdown formatting.
   - If `text` is specified or any other unwanted format, it defaults to plain text.

### Usage:
You can now call the `chat` function and pass the desired output format. It defaults to plain text unless specified otherwise.

In [6]:
lamma_client = LLMClient(lamma_config)
display(Markdown(lamma_client.chat(user_prompt=user_prompt, system_prompt=system_prompt)))

**Modifying the LLMClient Class to Support JSON Output and Markdown Formatting**

To add functionality to the `LLMClient` class to support JSON output and markdown formatting, you can make the following modifications:

### 1. Adding an `output_format` Parameter to the `Chat` Method

Firstly, let's modify the `chat` method to accept an additional parameter, `output_format`, which will determine how the response is formatted.

```python
def chat(
        self,
        user_prompt: str,
        system_prompt: Optional[str]=None,
        messages: Optional[List[dict]]=None,
        stream: bool = False,
        output_format: str = "text"
) -> Union[str, List[str], Dict]:
```

### 2. Defining the Meaning of `output_format` Parameters

We'll define a dictionary to map the supported output formats:

```python
_output_formats = {
    "json": {"format": "object", "indent_char": " "},
    "markdown": {"format": "plain_text", "type_chose": False}
}
```

### 3. Formatting the Response based on `output_format` Parameter

Now, let's modify the `chat` method to return a formatted response based on the `output_format` parameter:

```python
def chat(
        self,
        user_prompt: str,
        system_prompt: Optional[str]=None,
        messages: Optional[List[dict]]=None,
        stream: bool = False,
        output_format: str = "text"
) -> Union[str, List[str], Dict]:
    
    if not hasattr(self.client, 'get_response'):
        raise NotImplementedError(
            f"LLM Client does not support {output_format.lower()} format.")
        
    # ... (existing code remains the same)
```

### 4. Implementing JSON Output
```python
def chat(...) -> Union[str, List[str], Dict]:
    
    response = self.client.chat.completions.create(
        model=self.model,
        messages=chat_messages,
        stream=stream
    )
        
    def _json_formatting(response):
        data = {
            "message":response.choices[0].message.content
        }
        
        if "choices" in response:
            choices = []
            for choice in response.choices[0].delta.content or "":
                choices.append(choice)
            
            data["choices"] = choices
            
        return _output_formats["json"].get("format")(data)
    
    output = response
    
    # If `output_format` is set to json, format the response accordingly.
    if output_format == "json":
        return _json_formatting(output)
        
    # If `output_format` is markdown or text, return the original response
    elif output_format in ["markdown", "text"]:
        return output
        
    else:
        raise NotImplementedError(
            f"LLM Client does not support {output_format.lower()} format.")
```

### 5. Implementing Markdown Formatting

To implement markdown formatting, we can use the `re` module for string replacement:

```python
def _markdown_formatting(response):
    text = str(response.choices[0].message.content.strip())
    
    # Convert HTML tags to markdown
    import re
    html_tags = re.findall(r"<.*?>", response.choices[0].delta.content)
    for tag in html_tags:
        text = text.replace(tag, '|')
    
    # Handle special characters.
    from mark_safe import mark_safe
    
    return mark_safe(text)

# Inside the JSON formatting function
def _json_formatting(response):
    data = {
        "message": response.choices[0].message.content
    }
        
    if "choices" in response:
        choices = []
        for choice in response.choices[0].delta.content or "":
            choices.append(choice)
        
        data["choices"] = choices
        
    return _output_formats["json"].get("format")(_markdown_formatting(data))
```

Now you have modified the `LLMClient` class to support JSON output and markdown formatting.